# Модуль 3. Разбор кейсов: Идентичность, соли и таутомеры

**Модуль 3. Автор/аффилиация:** автор курса уточняет перед публикацией. **Среда:** WSL/uv/Python 3.12. **Данные:** molecules.csv: 12 исходных записей, два ошибочных ввода, соль, ковалентный Cl и стереопара.

Построить проверяемую таблицу raw→parent и сохранить смысл химических различий.

**Workflow:** ID/raw → парсинг → явная политика → parent → журнал и контрпримеры.

**Результаты обучения:** объяснить вход/политику; воспроизвести расчёт и отрицательный контроль; ограничить научный вывод.

Оцените результат до запуска. После разбора переходите во [второй notebook](lab03_practice.ipynb). [Установка среды](../SETUP.md).

[Связанная глава](../../Module_theory/Module%203_compound%20and%20reaction-representation.md).

## 0. Подготовка и паспорт запуска

Запускайте ячейки сверху вниз после Restart Kernel. Путь Python должен указывать на проектную `.venv`; устанавливать пакеты в ячейке не нужно. Эта ячейка только определяет пути и создаёт отдельный каталог результатов разбора.

In [ ]:
from pathlib import Path
import sys, json, math, csv, time, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'Labs/chemistry.py').is_file())
for folder in [ROOT/'Stepik_course/practice', ROOT/'Labs']:
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))
DATA = ROOT/'Labs/datasets'
OUT = ROOT/'Labs/results/lab03/worked'
OUT.mkdir(parents=True, exist_ok=True)
started = time.perf_counter()
print('Python:', sys.executable)


## 1. Сохраняем raw и вычисляем parent

Входящая поставка содержит альтернативные SMILES, соль и повреждённые строки. ID связывает исходник с решением, поэтому журнал включает и invalid. Parent — результат выбранной политики, а не восстановленный физический образец: удаление противоиона и нейтрализация меняют представление. Вызов screen объединяет парсинг, стандартизацию и запись причины решения; реализация доступна в Labs/chemistry.py.

**Перед запуском:** Спрогнозируйте пары эквивалентных графов до запуска.

In [ ]:
from chemistry import screen, standardize, parse
raw = pd.read_csv(DATA/"molecules.csv", keep_default_na=False)
records = [{"id": row.id, **screen(row.smiles)} for row in raw.itertuples()]
report = pd.DataFrame(records); display(report)
report.to_csv(OUT/"curated.csv", index=False)


**После запуска:** сверьте промежуточные переменные и объясните, какое условие проверено. Сохраняем raw и вычисляем parent — часть протокола; успешная ячейка не оценивает самостоятельный ответ.

## 2. Контрпримеры к наивному удалению фрагментов

Эквивалентная сериализация должна давать один parent. При этом ковалентный Cl остаётся частью молекулы, а энантиомеры сохраняют различие. Проверки по ID не зависят от порядка строк. Простое разбиение SMILES по точке не заменяет химическую политику: размер, заряд и роль фрагмента требуют явного правила.

**Перед запуском:** Обработайте все строки, включая ошибки, не удаляя ID.

In [ ]:
by_id = {row["id"]:row for row in records}
assert len(records)==12 and sum(row["status"]=="invalid" for row in records)==2
assert by_id["M01"]["parent_smiles"]==by_id["M02"]["parent_smiles"]
assert by_id["M03"]["parent_smiles"]==by_id["M04"]["parent_smiles"]
assert "Cl" in by_id["M05"]["parent_smiles"]
assert by_id["M10"]["parent_smiles"]!=by_id["M11"]["parent_smiles"]


**После запуска:** сверьте промежуточные переменные и объясните, какое условие проверено. Контрпримеры к наивному удалению фрагментов — часть протокола; успешная ячейка не оценивает самостоятельный ответ.

## 3. Таутомеры как явное решение

Сначала сравниваем пару без канонизации таутомеров, затем с переключателем функции standardize. Различие режимов записывается в provenance. Даже совпавшие parent не разрешают объединить экспериментальную растворимость соли и кислоты: нужны форма вещества, pH, температура и метод измерения.

**Перед запуском:** Проверьте отдельно ковалентный Cl, стереопару и объединение таутомеров по переключателю.

In [ ]:
pair=["O=c1cccc[nH]1","Oc1ccccn1"]
assert standardize(pair[0]).parent_smiles != standardize(pair[1]).parent_smiles
assert standardize(pair[0],True).parent_smiles == standardize(pair[1],True).parent_smiles
artifact={"rows":len(records),"invalid":2,"policy":by_id["M01"]["policy"]}


**После запуска:** сверьте промежуточные переменные и объясните, какое условие проверено. Таутомеры как явное решение — часть протокола; успешная ячейка не оценивает самостоятельный ответ.

## 4. Артефакт и граница доказательства

12 строк на выходе; M08/M09 invalid; M01=M02 и M03=M04 по parent; M10≠M11.

Следующая ячейка сохраняет именно новый запуск и хеши входов. Отсутствующее вычисление нельзя заменять старым output. Время относится к этому компьютеру и не является обещанием производительности.

In [ ]:
record={'module':3,'scope':'worked reference execution','python':sys.executable,'elapsed_seconds':time.perf_counter()-started,'result':artifact}
(OUT/'benchmark.json').write_text(json.dumps(record,ensure_ascii=False,indent=2))
print(OUT/'benchmark.json')

## Источники и следующий шаг

1. Исходный вычислительный пример: [Labs/chemistry.py](../chemistry.py), [описание данных](../datasets/README.md).
2. [RDKit: Getting Started](https://www.rdkit.org/docs/GettingStartedInPython.html).
3. [Карта чтения](../../Literature/READING_MAP.md).

Перейдите к [lab03_practice.ipynb](lab03_practice.ipynb). Через неделю повторите первый кейс без просмотра ключа.